In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder \
    .appName("Indian Food Analysis") \
    .getOrCreate()

In [6]:
filepath='D:\BDA_0024\ABD_LAB\datasets\indian_food.csv'

In [7]:
df = spark.read.csv(filepath,header=True,inferSchema=True)

In [8]:
df.show()

+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|          name|         ingredients|      diet|prep_time|cook_time|flavor_profile| course|        state|
+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|    Balu shahi|Maida flour, yogu...|vegetarian|       45|       25|         sweet|dessert|  West Bengal|
|        Boondi|Gram flour, ghee,...|vegetarian|       80|       30|         sweet|dessert|    Rajasthan|
|Gajar ka halwa|Carrots, milk, su...|vegetarian|       15|       60|         sweet|dessert|       Punjab|
|        Ghevar|Flour, ghee, kewr...|vegetarian|       15|       30|         sweet|dessert|    Rajasthan|
|   Gulab jamun|Milk powder, plai...|vegetarian|       15|       40|         sweet|dessert|  West Bengal|
|        Imarti|Sugar syrup, lent...|vegetarian|       10|       50|         sweet|dessert|  West Bengal|
|        Jalebi|Maida, corn flour...|vegetaria

In [9]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: integer (nullable = true)
 |-- cook_time: integer (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [13]:
# 1. Find out how many unique dishes are present. 

unique_dishes =df.select('name').distinct().count()

print("Number of unique dishes:", unique_dishes)

Number of unique dishes: 255


In [17]:
# 2. Which state has more dishes? 

state_count =df.groupBy('state').count().orderBy(desc('count'))
state_count.collect()[0]

Row(state='Gujarat', count=35)

In [25]:
# 3. How many dishes from state Karnataka? 

karnataka_count = df.filter(col('state') == 'Karnataka').count()

print("Number of dishes from Karnataka:", karnataka_count)

Number of dishes from Karnataka: 6


In [38]:
# 4. List number of unique regions 

unique_states = df.select('state').distinct().count()

print("Number of unique states:", unique_states)

Number of unique states: 25


In [43]:
# 5. Count number of dishes from each region.

state_count = df.groupBy('state').count().orderBy(desc('count'))

state_count.show(truncate=False)


+---------------+-----+
|state          |count|
+---------------+-----+
|Gujarat        |35   |
|Punjab         |32   |
|Maharashtra    |30   |
|-1             |24   |
|West Bengal    |24   |
|Assam          |21   |
|Tamil Nadu     |20   |
|Andhra Pradesh |10   |
|Uttar Pradesh  |9    |
|Kerala         |8    |
|Odisha         |7    |
|Karnataka      |6    |
|Rajasthan      |6    |
|Telangana      |5    |
|Goa            |3    |
|Bihar          |3    |
|Madhya Pradesh |2    |
|Manipur        |2    |
|Jammu & Kashmir|2    |
|Nagaland       |1    |
+---------------+-----+
only showing top 20 rows



In [46]:
# 6. List unique 'flavor_profile' and 'course' 

df.select('flavor_profile').distinct().show()

+--------------+
|flavor_profile|
+--------------+
|            -1|
|         spicy|
|         sweet|
|          sour|
|        bitter|
+--------------+



In [47]:
df.select('course').distinct().show()

+-----------+
|     course|
+-----------+
|    starter|
|    dessert|
|      snack|
|main course|
+-----------+



In [52]:
# 7. Which state has more 'main course'? 

main_course = df.filter(col('course') == 'main course').groupBy('state').count().orderBy(desc('count'))

main_course.show(50,truncate=False)

+---------------+-----+
|state          |count|
+---------------+-----+
|Punjab         |28   |
|Tamil Nadu     |17   |
|Assam          |15   |
|Gujarat        |12   |
|Maharashtra    |12   |
|-1             |9    |
|West Bengal    |9    |
|Kerala         |5    |
|Karnataka      |4    |
|Rajasthan      |3    |
|Uttar Pradesh  |3    |
|Bihar          |2    |
|Nagaland       |1    |
|Odisha         |1    |
|Madhya Pradesh |1    |
|Manipur        |1    |
|Jammu & Kashmir|1    |
|Goa            |1    |
|Haryana        |1    |
|NCT of Delhi   |1    |
|Telangana      |1    |
|Tripura        |1    |
+---------------+-----+



In [56]:
# 8. Give the %of dishes from each region.

total_dishes = df.count()

state_percentage = df \
    .groupBy('state') \
    .count() \
    .withColumn(
        'Percentage',
        round((col('count') / total_dishes) * 100, 2)
    ) \
    .orderBy(desc('Percentage'))

state_percentage.show(26)

+---------------+-----+----------+
|          state|count|Percentage|
+---------------+-----+----------+
|        Gujarat|   35|     13.73|
|         Punjab|   32|     12.55|
|    Maharashtra|   30|     11.76|
|             -1|   24|      9.41|
|    West Bengal|   24|      9.41|
|          Assam|   21|      8.24|
|     Tamil Nadu|   20|      7.84|
| Andhra Pradesh|   10|      3.92|
|  Uttar Pradesh|    9|      3.53|
|         Kerala|    8|      3.14|
|         Odisha|    7|      2.75|
|      Karnataka|    6|      2.35|
|      Rajasthan|    6|      2.35|
|      Telangana|    5|      1.96|
|            Goa|    3|      1.18|
|          Bihar|    3|      1.18|
| Madhya Pradesh|    2|      0.78|
|        Manipur|    2|      0.78|
|Jammu & Kashmir|    2|      0.78|
|       Nagaland|    1|      0.39|
|   Chhattisgarh|    1|      0.39|
|        Haryana|    1|      0.39|
|   NCT of Delhi|    1|      0.39|
|        Tripura|    1|      0.39|
|    Uttarakhand|    1|      0.39|
+---------------+---

In [ ]:
# 9. List the states which has more dishes from each region.